In [ ]:
import tensorflow as tf
import pandas as pd
import seaborn as sns
from tensorflow.keras.layers import Dense,Normalization,InputLayer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
import matplotlib.pyplot as plt

In [ ]:
data=pd.read_csv("train.csv")
data.head()


FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
sns.pairplot(data[['years','km','rating','condition','economy','top speed','hp','torque',
                   'current price']],diag_kind='kde')


In [ ]:
data_tensor=tf.constant(data)
data_tensor=tf.cast(data_tensor,tf.float32)
data_tensor=tf.random.shuffle(data_tensor)
data_tensor

In [ ]:
X=data_tensor[:,3:-1]
y=data_tensor[:,-1]
y=tf.expand_dims(y,axis=-1)
y[:5]

In [ ]:
normalizer=tf.keras.layers.Normalization()



In [ ]:
TRAIN=0.6 *len(X)
TEST=0.2*len(X)
VAL=0.2*len(X)

X_train=X[:int(TRAIN)]
y_train=y[:int(TRAIN)]

X_val=X[int(TRAIN):int(TRAIN+VAL)]
y_val=y[int(TRAIN):int(TRAIN+VAL)]

X_test=X[int(TRAIN+VAL):]
y_test=y[int(TRAIN+VAL):]
X_train.shape,y_train.shape,X_val.shape,y_val.shape,X_test.shape,y_test.shape

In [ ]:
train_data_set=tf.data.Dataset.from_tensor_slices((X_train,y_train))
train_data_set=train_data_set.shuffle(buffer_size=8).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
val_data_set=tf.data.Dataset.from_tensor_slices((X_val,y_val))
val_data_set=val_data_set.shuffle(buffer_size=8).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
test_data_set=tf.data.Dataset.from_tensor_slices((X_test,y_test))
test_data_set=test_data_set.shuffle(buffer_size=8).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
normalizer.adapt(X_train)
model=tf.keras.Sequential([
    InputLayer(input_shape=(8,)),
    normalizer,
    Dense(32,activation='relu'),
    Dense(32,activation='relu'),
    Dense(1),

])
model.compile(optimizer=Adam(),loss='mae',metrics=[RootMeanSquaredError()])
tf.keras.utils.plot_model(model,to_file="model.png",show_shapes=True)

In [ ]:
history=model.fit(train_data_set, epochs=100, validation_data=(val_data_set), verbose=1)

In [ ]:
history
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['loss','val_loss'])
plt.show()

In [ ]:
y_pred=model.predict(X_test)
print((abs(y_test-y_pred)/y_test*100)[:5])